In [1]:
from pathlib import Path ## Rutas relativas
from PIL import Image  ## Abrir y manipular imagenes 
import numpy as np ### Procesar Matrices - Valores de pixeles
import matplotlib.pyplot as plt ## Graficar
import cv2
import numpy as np

**Abrir las imagenes**

In [2]:
IMAGE_DIR = Path("data/images")  ##Ruta a la carpeta donde guardamos las imagenes

image_paths = sorted(
    path for path in IMAGE_DIR.glob("*") ## Lista de rutas de archivos
    if path.is_file()
)

print(f"Cant. de imagenes: {len(image_paths)}")

Cant. de imagenes: 10


**Conocer las imagenes**


In [3]:
for path in image_paths:
    if path.is_file():
        image = Image.open(path) ## abrir imagen
        width, height = image.size ## Conocer el tama;o de la imagen

        print(f"{path.name}: tamaño={width} × {height}")

albert-bloch_figures-in-silver-light.jpg: tamaño=1657 × 1382
albert-gleizes_femme-cubiste-1921.jpg: tamaño=1382 × 1889
aldemir-martins_baiana-1980.jpg: tamaño=1382 × 1845
aldo-mondino_collage-1973.jpg: tamaño=1382 × 1742
aleksey-savrasov_early-spring-2.jpg: tamaño=1880 × 1382
alvaro-lapa_c-line-s-notebook-1990.jpg: tamaño=3300 × 1382
andre-derain_the-port-of-collioure-1905.jpg: tamaño=1695 × 1382
betty-parsons_ladder-1968.jpg: tamaño=1382 × 1949
joshua-reynolds_jane-fleming-later-countess-of-harrington-1779.jpg: tamaño=1382 × 2289
yayoi-kusama_fields-in-spring-1988.jpg: tamaño=1382 × 1629


**Redimensionamiento de las imágenes**

Las imágenes seleccionadas tienen dimensiones variables, por ejemplo, `3300 × 1382` y `1382 × 1629`. Por esta razón, se realizará un proceso de redimensionamiento (*resize*) antes de utilizarlas en el modelo.

No se utilizará un tamaño fijo para todas las imágenes, ya que esto podría cambiar su relación de aspecto y generar deformaciones. En su lugar, se establecerá un tamaño máximo y cada imagen se redimensionará manteniendo su proporción original.

El objetivo del redimensionamiento es **reducir el número de píxeles que debe procesar K-Means**, haciendo que el algoritmo sea más rápido y requiera menos recursos, pero conservando la información de color de las obras.

Aunque en este ejercicio se trabaja con un número reducido de imágenes, algunas de ellas tienen millones de píxeles. Además, el pipeline debe estar preparado para recibir imágenes nuevas que podrían tener una resolución mucho mayor. Por esta razón, se incluye el redimensionamiento como parte del preprocesamiento, buscando reducir el costo computacional sin perder información relevante para la extracción de la paleta de colores.



**Decisión sobre el factor de reducción**


En lugar de establecer un tamaño fijo para todas las imágenes, se decidió evaluar diferentes **factores de reducción**. Esta decisión permite mantener la proporción original de cada imagen y, al mismo tiempo, analizar cómo la reducción del número de píxeles afecta la información de color y el costo computacional del procesamiento con K-Means. Se probarán los factores **1, 1/2, 1/4, 1/8, 1/16 y 1/32**, donde cada factor se aplica tanto al ancho como al alto de la imagen. De esta forma, la selección del factor final no se realizará de manera arbitraria, sino a partir de la comparación de los resultados obtenidos en términos de conservación de la información y eficiencia computacional.


In [4]:
## Definir la funcion para conocer los nuevos tama;os segun el factor para redimensionar.

RESIZE_FACTORS = [1, 1/2, 1/3, 1/4, 1/8, 1/16, 1/32]

def calculate_new_size(width, height, factor):
    new_width = int(width * factor)
    new_height = int(height * factor)

    return new_width, new_height

## Aplicar la funcion a las imagenes 

for path in image_paths:
    image = Image.open(path)
    width, height = image.size

    for factor in RESIZE_FACTORS:
        new_width, new_height = calculate_new_size(
            width, height, factor
        )

**Metodo del area y Lanczos**

In [5]:
## Para cada imagen aplicar el factor de reduccion y los dos metodos

for path in image_paths:
    image = Image.open(path)
    image = np.array(image)

    height, width = image.shape[:2]

    for factor in RESIZE_FACTORS:
        new_width, new_height = calculate_new_size(
            width, height, factor
        )

        resized_area = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA
        )



Todas las imágenes utilizadas se encuentran en formato RGB. Por lo tanto, no sería necesario realizar una conversión de color sobre este conjunto de datos. Sin embargo, se incluirá la conversión a RGB como parte del pipeline de preprocesamiento, con el propósito de garantizar que cualquier imagen incorporada posteriormente sea transformada a una representación consistente de tres canales antes de ingresar al modelo.

Adicionalmente, cada imagen se transforma de una matriz tridimensional a una matriz bidimensional mediante un reshape, ya que K-Means requiere que cada observación esté representada como una fila independiente. Los valores de los tres canales se normalizan dividiéndolos entre 255.

In [6]:
images_rgb = []

for path in image_paths:
    if path.is_file():
        image_bgr = cv2.imread(str(path))
        
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        images_rgb.append(image_rgb)

for path, image in zip(image_paths, images_rgb):
    print(path.name, f"Formato: RGB (canales: {image.shape[2]})")

albert-bloch_figures-in-silver-light.jpg Formato: RGB (canales: 3)
albert-gleizes_femme-cubiste-1921.jpg Formato: RGB (canales: 3)
aldemir-martins_baiana-1980.jpg Formato: RGB (canales: 3)
aldo-mondino_collage-1973.jpg Formato: RGB (canales: 3)
aleksey-savrasov_early-spring-2.jpg Formato: RGB (canales: 3)
alvaro-lapa_c-line-s-notebook-1990.jpg Formato: RGB (canales: 3)
andre-derain_the-port-of-collioure-1905.jpg Formato: RGB (canales: 3)
betty-parsons_ladder-1968.jpg Formato: RGB (canales: 3)
joshua-reynolds_jane-fleming-later-countess-of-harrington-1779.jpg Formato: RGB (canales: 3)
yayoi-kusama_fields-in-spring-1988.jpg Formato: RGB (canales: 3)


In [7]:
def flatten_and_normalize(image):
    
    img_array = np.asarray(image, dtype=np.float32)
    image_flat = img_array.reshape(-1, 3) / 255.0
    return image_flat